# Load Balancer V2 — SQLite State Database Explorer

This notebook lets you explore the SQLite database used by `load_balancer_v2.py` to store request/response logs and conversation affinity state.

## 1. Setup

In [2]:
import sqlite3
import json
import pandas as pd
from pathlib import Path

# Find the database file — update DB_PATH if your setup differs
DB_PATH = Path.home() / ".config" / "llm-proxy" / "state.db"

if not DB_PATH.exists():
    # Try to detect from environment or common locations
    import os
    for candidate in [
        Path.home() / ".config" / "llm-proxy.yaml",
        Path("~/.config/llm-proxy.yaml").expanduser(),
        Path("/etc/llm-proxy.yaml"),
    ]:
        if candidate.exists():
            print(f"Found config at {candidate}")
            break

print(f"Database path: {DB_PATH}")
print(f"Exists: {DB_PATH.exists()}")

Found config at /home/anhvth8/.config/llm-proxy.yaml
Database path: /home/anhvth8/.config/llm-proxy/state.db
Exists: False


## 2. Helper: connect and convenience query

In [3]:
def connect(readonly=True):
    """Connect to the database."""
    uri = f"file:{DB_PATH}?mode=ro" if readonly else str(DB_PATH)
    return sqlite3.connect(uri, uri=True)

def q(sql, params=None):
    """Execute a query and return a DataFrame."""
    with connect() as conn:
        return pd.read_sql(sql, conn, params=params)

def qraw(sql, params=None):
    """Execute a query and return raw rows (for JSON fields)."""
    with connect() as conn:
        conn.row_factory = sqlite3.Row
        cursor = conn.execute(sql, params or [])
        return cursor.fetchall()

## 3. Schema overview

In [4]:
# List all tables
tables = q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
print("Tables:")
for t in tables["name"]:
    print(f"  {t}")

OperationalError: unable to open database file

In [5]:
# Show CREATE statement for each table
for t in tables["name"]:
    print(f"\n=== {t} ===")
    schema = q(f"SELECT sql FROM sqlite_master WHERE name = '{t}'")
    print(schema["sql"].iloc[0])

NameError: name 'tables' is not defined

## 4. conversations — conversation metadata

In [6]:
q("SELECT * FROM conversations ORDER BY last_seen_ns DESC LIMIT 20")

OperationalError: unable to open database file

## 5. requests — the main log table (input + output payloads)

In [7]:
q("SELECT request_id, conversation_id, endpoint_used, route_reason, status_code, created_ns, collected_at_ns FROM requests ORDER BY request_id DESC LIMIT 20")

OperationalError: unable to open database file

### 5a. Inspect a single request's input and output

In [8]:
# Pick a request_id (change this to explore a different one)
REQUEST_ID = 1

row = qraw(
    "SELECT conversation_id, request_meta_json, response_json, input_state_hash FROM requests WHERE request_id = ?",
    [REQUEST_ID],
)[0]

request_meta = json.loads(row["request_meta_json"])
response_payload = json.loads(row["response_json"])

print(f"Conversation saved: yes, conversation_id={row['conversation_id']}")
print("=== Request Meta ===")
print(json.dumps(request_meta, indent=2, ensure_ascii=False))

OperationalError: unable to open database file

In [9]:
print("=== Response Payload ===")
print(json.dumps(response_payload, indent=2, ensure_ascii=False))

=== Response Payload ===


NameError: name 'response_payload' is not defined

### 5b. Reconstruct full messages from state_hash

In [10]:
def reconstruct_messages(state_hash, conn=None):
    """Reconstruct the full message list for a given state_hash."""
    should_close = conn is None
    conn = conn or connect()
    try:
        # Get the matched_prefix_hash to recurse
        state_row = conn.execute(
            "SELECT matched_prefix_hash FROM input_states WHERE state_hash = ?",
            (state_hash,),
        ).fetchone()
        if state_row is None:
            return []

        messages = []
        matched_prefix = state_row["matched_prefix_hash"]
        if isinstance(matched_prefix, str) and matched_prefix:
            messages.extend(reconstruct_messages(matched_prefix, conn))

        rows = conn.execute(
            """
            SELECT m.raw_json
            FROM state_tail_messages AS stm
            JOIN messages AS m ON m.message_hash = stm.message_hash
            WHERE stm.state_hash = ?
            ORDER BY stm.ordinal ASC
            """,
            (state_hash,),
        ).fetchall()
        for r in rows:
            messages.append(json.loads(r["raw_json"]))
        return messages
    finally:
        if should_close:
            conn.close()


messages = reconstruct_messages(row["input_state_hash"])
print(f"Total messages: {len(messages)}")
for i, msg in enumerate(messages):
    role = msg.get("role", "?")
    content = str(msg.get("content", ""))[:120]
    print(f"  [{i}] {role}: {content}")

NameError: name 'row' is not defined

## 6. input_states — conversation prefixes

In [11]:
q("SELECT * FROM input_states ORDER BY last_seen_ns DESC LIMIT 20")

OperationalError: unable to open database file

## 7. messages — raw message JSON blobs

In [12]:
q("SELECT message_hash, raw_json, created_ns FROM messages ORDER BY created_ns DESC LIMIT 10")

OperationalError: unable to open database file

## 8. route_affinity — loose-hash based routing

In [13]:
q("SELECT * FROM route_affinity ORDER BY last_seen_ns DESC LIMIT 20")

OperationalError: unable to open database file

## 9. Statistics

In [14]:
# Row counts
for table in tables["name"]:
    cnt = q(f"SELECT COUNT(*) as n FROM {table}")
    print(f"{table}: {cnt.iloc[0]['n']} rows")

NameError: name 'tables' is not defined

In [15]:
# Requests per route_reason
q("""
SELECT route_reason, COUNT(*) as count, AVG(status_code) as avg_status
FROM requests
GROUP BY route_reason
ORDER BY count DESC
""")

OperationalError: unable to open database file

In [16]:
# Requests per endpoint
q("""
SELECT endpoint_used, COUNT(*) as count
FROM requests
GROUP BY endpoint_used
ORDER BY count DESC
""")

OperationalError: unable to open database file

In [ ]:
# Uncollected requests (not yet processed by a collector)
q("SELECT COUNT(*) as uncollected FROM requests WHERE collected_at_ns IS NULL")

## 10. Export all requests to JSONL

In [ ]:
def export_to_jsonl(output_path="requests_export.jsonl"):
    """Export all requests with reconstructed input/output payloads."""
    rows = qraw(
        """
        SELECT request_id, conversation_id, input_state_hash,
               request_meta_json, response_json, endpoint_used,
               route_reason, status_code, created_ns, collected_at_ns
        FROM requests
        ORDER BY request_id ASC
        """
    )
    with open(output_path, "w") as f:
        for r in rows:
            request_meta = json.loads(r["request_meta_json"])
            response_payload = json.loads(r["response_json"])
            messages = reconstruct_messages(r["input_state_hash"])
            record = {
                "request_id": r["request_id"],
                "conversation_id": r["conversation_id"],
                "input": {
                    **request_meta,
                    "messages": messages,
                },
                "output": response_payload,
                "endpoint_used": r["endpoint_used"],
                "route_reason": r["route_reason"],
                "status_code": r["status_code"],
                "created_ns": r["created_ns"],
                "collected_at_ns": r["collected_at_ns"],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Exported {len(rows)} records to {output_path}")

# Uncomment to run:
# export_to_jsonl()